# EDA on raw data marathos

In [0]:
VOLUME_PATH = "/Volumes/marathos/default/raw"

spark.sql(f"LIST '{VOLUME_PATH}'").display()

In [0]:
%sql
SHOW SCHEMAS IN supply_chain_demo;

In [0]:
%sql

FROM marathos.bronze.raw_marathos LIMIT 5;

In [0]:
df = spark.read.format("csv").options(header=True, inferSchema=True).load(f"{VOLUME_PATH}/data/TWO_CENTURIES_OF_UM_RACES.csv")

print(f"Number of Rows: {df.count():,}")
print(f"Number of Columns: {len(df.columns)}")

df.printSchema()

In [0]:
display(df.summary())

In [0]:

from pyspark.sql.functions import col, sum as spark_sum

null_counts = df.select(
    [spark_sum(col(column).isNull().cast("int")).alias(column) for column in df.columns]
)

null_counts = null_counts.collect()[0].asDict()

[(column, nulls) for column, nulls in null_counts.items() if nulls > 0]

In [0]:
from pyspark.sql.functions import col, countDistinct, concat_ws

unique_race_instances = df.select(
    countDistinct(
        concat_ws(" - ", col("Year of event"), col("Event name"))
    ).alias("total_unique_races")
)

display(unique_race_instances)

In [0]:
from pyspark.sql.functions import col, countDistinct

recurring_events = df.groupBy("Event name") \
                     .agg(countDistinct("Year of event").alias("years_held")) \
                     .filter(col("years_held") > 1) \
                     .orderBy(col("years_held").desc())

display(recurring_events)

In [0]:
%sql
SELECT `Athlete age category`, COUNT(*) as total_records
FROM marathos.bronze.raw_marathos
WHERE `Athlete age category` IS NOT NULL
GROUP BY `Athlete age category`
ORDER BY total_records DESC

In [0]:
%sql

SELECT * FROM marathos.bronze.raw_marathos 
WHERE `Athlete age category` = 'MU20'

In [0]:
%sql
SELECT 
    `Year of event`,
    COUNT(*) as total_runners,
    SUM(CASE WHEN `Athlete age category` IS NULL THEN 1 ELSE 0 END) as null_count,
    ROUND((SUM(CASE WHEN `Athlete age category` IS NULL THEN 1 ELSE 0 END) / COUNT(*)) * 100, 2) as null_percentage
FROM marathos.bronze.raw_marathos
GROUP BY `Year of event`
ORDER BY `Year of event` ASC

In [0]:
%sql
SELECT 
    `Athlete country`,
    COUNT(*) as total_runners,
    SUM(CASE WHEN `Athlete age category` IS NULL THEN 1 ELSE 0 END) as null_count,
    ROUND((SUM(CASE WHEN `Athlete age category` IS NULL THEN 1 ELSE 0 END) / COUNT(*)) * 100, 2) as null_percentage
FROM marathos.bronze.raw_marathos
GROUP BY `Athlete country`
HAVING total_runners > 1000 
ORDER BY null_percentage DESC

In [0]:
%sql
SELECT `Year of event`, `Event name`, `Athlete performance`, `Athlete year of birth`, `Athlete age category`
FROM marathos.bronze.raw_marathos
WHERE `Athlete age category` IS NULL
LIMIT 20

In [0]:
%sql
SELECT 
    `Athlete country`, 
    COUNT(*) AS runner_count,
    ROUND(COUNT(*) * 100.0 / SUM(COUNT(*)) OVER(), 2) AS percentage_of_global_total
FROM marathos.bronze.raw_marathos
WHERE `Athlete country` IS NOT NULL AND `Athlete country` != ''
GROUP BY `Athlete country`
ORDER BY runner_count DESC
LIMIT 15